# PPL Comparison Figures

This notebook regenerates all paper-facing PPL comparison figures for LLaMA-2-7B and the
four-model Vanilla-BFP sweep. Each section writes a 600-DPI PNG and a same-stem vector PDF
under `figures/`.

- **Section 1** — BFP / BiE / Activation BiE / Top-2 delta PPL (`figures/g16/`)
- **Section 2** — Symmetric Hybrid DEWA threshold sweep, log + zoom (`figures/g16/`)
- **Section 3** — Selected `(T_skip, T_replace)` gallery vs. Top-2 (`figures/g16/`)
- **Section 4** — Four-model Vanilla-BFP delta PPL for G16 and G32 (`figures/g16_g32/`)

Run all cells from this directory so relative paths resolve correctly.

In [1]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FormatStrFormatter, MaxNLocator, MultipleLocator

PLOT_DIR = Path(".").resolve()
RESEARCH_ROOT = PLOT_DIR.parents[1]
EXPERIMENTS_ROOT = RESEARCH_ROOT / "experiments"
FIGURES_G16 = PLOT_DIR / "figures" / "g16"
FIGURES_G16_G32 = PLOT_DIR / "figures" / "g16_g32"
GROUP_SIZE = 16
FORMATS = tuple(range(4, 9))


def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def validate_protocol(
    payload: dict[str, Any],
    baseline: dict[str, Any],
    path: Path,
) -> None:
    for key in (
        "model",
        "dataset",
        "split",
        "evaluation_protocol",
        "context_length",
        "stride",
        "drop_remainder",
        "evaluated_tokens",
    ):
        if payload.get(key) != baseline.get(key):
            raise ValueError(f"Protocol mismatch for '{key}' in {path}")


def configure_ieee_style() -> None:
    plt.rcParams.update(
        {
            "font.family": "serif",
            "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
            "font.size": 8,
            "axes.labelsize": 8,
            "xtick.labelsize": 7,
            "ytick.labelsize": 7,
            "legend.fontsize": 7,
            "axes.linewidth": 0.8,
            "xtick.major.width": 0.8,
            "ytick.major.width": 0.8,
            "xtick.major.size": 3,
            "ytick.major.size": 3,
            "mathtext.fontset": "stix",
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "savefig.facecolor": "white",
        }
    )


def save_figure(fig: plt.Figure, stem: Path) -> Path:
    stem.parent.mkdir(parents=True, exist_ok=True)
    png_path = stem.with_suffix(".png")
    fig.savefig(png_path, dpi=600, bbox_inches="tight")
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {png_path}")
    return png_path


def style_axes(ax: plt.Axes, *, grid_style: str = "--") -> None:
    ax.grid(axis="y", color="0.84", linewidth=0.55, linestyle=grid_style, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="in", top=False, right=False)


configure_ieee_style()

## 1. Method comparison (`ppl_delta_comparison`)

Compare Vanilla BFP, BiE, Activation BiE, and Activation BiE Top-2 against the FP16 baseline.

In [2]:
@dataclass(frozen=True)
class MethodSpec:
    label: str
    relative_dir: Path
    config_key: str
    color: str
    marker: str
    linestyle: str


@dataclass(frozen=True)
class MethodResultPoint:
    bits: int
    perplexity: float
    quantize_lm_head: bool
    path: Path


METHODS = (
    MethodSpec("BFP", Path("02_Vanilla_BFP") / "llama2-7b", "bfp_config", "#4C78A8", "o", "-"),
    MethodSpec("BiE", Path("06_BiE") / "llama2-7b", "bie_config", "#D17C2F", "s", "-"),
    MethodSpec(
        "Activation BiE",
        Path("07_ActivationBiE") / "llama2-7b",
        "hybrid_config",
        "#59A14F",
        "^",
        "-",
    ),
    MethodSpec(
        "Activation BiE Top-2",
        Path("08_ActivationBiETop2") / "llama2-7b",
        "hybrid_config",
        "#B279A2",
        "v",
        "-",
    ),
)


def discover_method_results(
    spec: MethodSpec,
    baseline: dict[str, Any],
) -> dict[int, MethodResultPoint]:
    result_dir = EXPERIMENTS_ROOT / spec.relative_dir
    candidates: dict[int, list[MethodResultPoint]] = {}
    baseline_ppl = float(baseline["perplexity"])

    for path in sorted(result_dir.rglob("*.json")):
        payload = load_json(path)
        config = payload.get(spec.config_key)
        if not isinstance(config, dict):
            continue
        if int(config.get("block_size", -1)) != GROUP_SIZE:
            continue
        bits = 1 + int(config.get("mantissa_bits", -1))
        if bits not in FORMATS:
            continue
        validate_protocol(payload, baseline, path)
        reference = float(payload.get("baseline_perplexity", np.nan))
        if not np.isclose(reference, baseline_ppl, rtol=0.0, atol=1e-12):
            raise ValueError(f"FP16 baseline mismatch in {path}")
        perplexity = float(payload["perplexity"])
        if not np.isfinite(perplexity) or perplexity <= 0.0:
            raise ValueError(f"Invalid perplexity in {path}")
        candidates.setdefault(bits, []).append(
            MethodResultPoint(
                bits=bits,
                perplexity=perplexity,
                quantize_lm_head=bool(config.get("quantize_lm_head", False)),
                path=path,
            )
        )

    selected: dict[int, MethodResultPoint] = {}
    for bits, points in candidates.items():
        points.sort(key=lambda point: (point.quantize_lm_head, str(point.path)))
        preferred_scope = points[0].quantize_lm_head
        same_scope = [
            point for point in points if point.quantize_lm_head == preferred_scope
        ]
        if len(same_scope) > 1:
            paths = ", ".join(str(point.path) for point in same_scope)
            raise ValueError(f"Ambiguous {spec.label}{bits} results: {paths}")
        selected[bits] = points[0]
    return selected


baseline = load_json(
    EXPERIMENTS_ROOT / "01_Baseline" / "llama2-7b" / "baseline.json"
)
baseline_ppl = float(baseline["perplexity"])
method_series = {
    spec.label: discover_method_results(spec, baseline) for spec in METHODS
}

x = np.arange(len(FORMATS), dtype=np.float64)
finite_values: list[float] = []
fig, ax = plt.subplots(figsize=(3.5, 2.35))
for spec in METHODS:
    values = np.full(len(FORMATS), np.nan, dtype=np.float64)
    for index, bits in enumerate(FORMATS):
        point = method_series[spec.label].get(bits)
        if point is not None:
            values[index] = point.perplexity - baseline_ppl
            finite_values.append(float(values[index]))
    ax.plot(
        x,
        values,
        color=spec.color,
        linewidth=1.2,
        linestyle=spec.linestyle,
        marker=spec.marker,
        markersize=4.0,
        markerfacecolor=spec.color,
        markeredgecolor="0.20",
        markeredgewidth=0.55,
        label=spec.label,
        zorder=3,
    )

ax.set_xlabel("Data Format")
ax.set_ylabel(r"$\Delta$PPL (vs. FP16)")
ax.set_xticks(x, [f"BFP{bits}" for bits in FORMATS])
ax.set_xlim(-0.18, len(FORMATS) - 0.82)
ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
style_axes(ax)
ax.legend(frameon=False, loc="upper right", handlelength=2.0)
minimum = min(finite_values)
maximum = max(finite_values)
lower = min(0.0, minimum)
margin = max(0.03, 0.10 * (maximum - lower))
ax.set_ylim(lower - (margin if lower < 0.0 else 0.0), maximum + margin)
fig.tight_layout(pad=0.35)
save_figure(fig, FIGURES_G16 / "ppl_delta_comparison")

Saved: C:\Users\user\Desktop\research\research1\plot\03_ppl-comparison\figures\g16\ppl_delta_comparison.png


WindowsPath('C:/Users/user/Desktop/research/research1/plot/03_ppl-comparison/figures/g16/ppl_delta_comparison.png')

## 2. Symmetric Hybrid DEWA sweep (`hybrid_symmetric_ppl_log_zoom_t3-12`)

Full symmetric threshold sweep on log PPL with a linear zoom for $T = 7$ through $12$.

In [3]:
EXPECTED_SYMMETRIC_THRESHOLDS = tuple(range(3, 13))
SYMMETRIC_RESULT_PATTERN = re.compile(
    r"w-bfp4-a-bie4-top2cap-dewa-fpacc-excskip-t(?P<threshold>\d+)-s2048\.json"
)


@dataclass(frozen=True)
class SymmetricSweepPoint:
    threshold: int
    perplexity: float
    path: Path


def discover_symmetric_sweep() -> tuple[float, dict[int, SymmetricSweepPoint]]:
    result_dir = EXPERIMENTS_ROOT / "09_Hybrid" / "llama2-7b"
    fp16_baseline = float(baseline["perplexity"])
    top2_baseline: float | None = None
    points: dict[int, SymmetricSweepPoint] = {}

    for path in sorted(result_dir.glob("*.json")):
        match = SYMMETRIC_RESULT_PATTERN.fullmatch(path.name)
        if match is None:
            continue
        threshold = int(match.group("threshold"))
        if threshold not in EXPECTED_SYMMETRIC_THRESHOLDS:
            continue

        payload = load_json(path)
        hybrid_config = payload.get("hybrid_config")
        dewa_config = payload.get("dewa_config")
        if not isinstance(hybrid_config, dict) or not isinstance(dewa_config, dict):
            raise ValueError(f"Missing Hybrid/DEWA configuration in {path}")
        if int(hybrid_config.get("block_size", -1)) != GROUP_SIZE:
            raise ValueError(f"Expected Group-16 configuration in {path}")
        if 1 + int(hybrid_config.get("mantissa_bits", -1)) != 4:
            raise ValueError(f"Expected BFP4/BiE4 mantissa format in {path}")

        normal_threshold = int(dewa_config.get("threshold_bits", -1))
        exception_threshold = int(dewa_config.get("exception_threshold_bits", -1))
        if (normal_threshold, exception_threshold) != (threshold, threshold):
            raise ValueError(f"Expected symmetric T={threshold} in {path}")

        validate_protocol(payload, baseline, path)
        fp16_reference = float(payload.get("fp16_baseline_perplexity", np.nan))
        if not np.isclose(fp16_reference, fp16_baseline, rtol=0.0, atol=1e-12):
            raise ValueError(f"FP16 baseline mismatch in {path}")

        point_top2_baseline = float(payload.get("top2_baseline_perplexity", np.nan))
        if not np.isfinite(point_top2_baseline):
            raise ValueError(f"Missing Top-2 baseline in {path}")
        if top2_baseline is None:
            top2_baseline = point_top2_baseline
        elif not np.isclose(point_top2_baseline, top2_baseline, rtol=0.0, atol=1e-12):
            raise ValueError(f"Top-2 baseline mismatch in {path}")

        perplexity = float(payload["perplexity"])
        if threshold in points:
            raise ValueError(f"Duplicate symmetric threshold T={threshold}")
        points[threshold] = SymmetricSweepPoint(threshold, perplexity, path)

    missing = sorted(set(EXPECTED_SYMMETRIC_THRESHOLDS) - set(points))
    if missing:
        raise ValueError(f"Missing symmetric DEWA threshold results: {missing}")
    assert top2_baseline is not None
    return top2_baseline, points


top2_baseline, symmetric_points = discover_symmetric_sweep()
thresholds = np.asarray(EXPECTED_SYMMETRIC_THRESHOLDS, dtype=np.int64)
perplexities = np.asarray(
    [symmetric_points[int(threshold)].perplexity for threshold in thresholds],
    dtype=np.float64,
)
zoom_thresholds = thresholds[thresholds >= 7]
zoom_perplexities = perplexities[thresholds >= 7]

fig, (overview_ax, zoom_ax) = plt.subplots(
    1,
    2,
    figsize=(7.16, 2.65),
    gridspec_kw={"width_ratios": (1.15, 1.0)},
)
overview_ax.plot(
    thresholds,
    perplexities,
    color="#4C78A8",
    linewidth=1.3,
    linestyle="-",
    marker="o",
    markersize=4.0,
    markerfacecolor="#4C78A8",
    markeredgecolor="0.20",
    markeredgewidth=0.55,
    label="Symmetric Hybrid DEWA",
    zorder=3,
)
overview_ax.axhline(
    top2_baseline,
    color="0.42",
    linewidth=0.9,
    linestyle="--",
    label="Top-2 A-BiE4 baseline",
    zorder=1,
)
overview_ax.set_yscale("log")
overview_ax.set_ylabel("Perplexity (PPL, log scale)")
overview_ax.set_xticks(thresholds)
overview_ax.set_xlim(float(thresholds.min()) - 0.25, float(thresholds.max()) + 0.25)
overview_ax.set_ylim(top2_baseline * 0.88, perplexities.max() * 1.8)
style_axes(overview_ax)
overview_ax.legend(frameon=False, loc="upper right", handlelength=1.8)

zoom_ax.plot(
    zoom_thresholds,
    zoom_perplexities,
    color="#4C78A8",
    linewidth=1.3,
    linestyle="-",
    marker="o",
    markersize=4.0,
    markerfacecolor="#4C78A8",
    markeredgecolor="0.20",
    markeredgewidth=0.55,
    zorder=3,
)
zoom_ax.axhline(top2_baseline, color="0.42", linewidth=0.9, linestyle="--", zorder=1)
zoom_ax.set_ylabel("Perplexity (PPL)")
zoom_ax.set_xticks(zoom_thresholds)
zoom_ax.set_xlim(float(zoom_thresholds.min()) - 0.15, float(zoom_thresholds.max()) + 0.15)
zoom_ax.set_ylim(6.00, 6.60)
zoom_ax.yaxis.set_major_locator(MultipleLocator(0.10))
zoom_ax.yaxis.set_minor_locator(MultipleLocator(0.02))
zoom_ax.tick_params(axis="y", which="minor", length=2)
style_axes(zoom_ax)
zoom_ax.text(
    0.98,
    0.94,
    r"Zoom: $T=7$--$12$ (linear scale)",
    transform=zoom_ax.transAxes,
    ha="right",
    va="top",
    fontsize=7,
)
fig.supxlabel(r"Symmetric DEWA threshold, $T$", y=0.01)
fig.tight_layout(rect=(0.0, 0.07, 1.0, 1.0), pad=0.35, w_pad=1.1)
save_figure(fig, FIGURES_G16 / "hybrid_symmetric_ppl_log_zoom_t3-12")

Saved: C:\Users\user\Desktop\research\research1\plot\03_ppl-comparison\figures\g16\hybrid_symmetric_ppl_log_zoom_t3-12.png


WindowsPath('C:/Users/user/Desktop/research/research1/plot/03_ppl-comparison/figures/g16/hybrid_symmetric_ppl_log_zoom_t3-12.png')

## 3. `(T_skip, T_replace)` gallery (`hybrid_tskip_treplace_delta_ppl`)

Selected design points: `(8, 8)`, `(9, 9)` through `(9, 2)`, then `(10, 10)`.

In [4]:
T9_REPLACE_THRESHOLDS = tuple(range(9, 1, -1))
DESIGN_POINTS = (
    (8, 8),
    *((9, replace_threshold) for replace_threshold in T9_REPLACE_THRESHOLDS),
    (10, 10),
)
HYBRID_DIR = Path("09_Hybrid") / "llama2-7b"
SYMMETRIC_NAME = "w-bfp4-a-bie4-top2cap-dewa-fpacc-excskip-t{threshold}-s2048.json"
ASYMMETRIC_NAME = (
    "w-bfp4-a-bie4-top2cap-asym-dewa-fpacc-tskip9-"
    "treplace{replace_threshold}-s2048.json"
)


@dataclass(frozen=True)
class DesignPoint:
    skip_threshold: int
    replace_threshold: int
    perplexity: float
    path: Path


def hybrid_result_path(skip_threshold: int, replace_threshold: int) -> Path:
    result_dir = EXPERIMENTS_ROOT / HYBRID_DIR
    if skip_threshold == replace_threshold:
        name = SYMMETRIC_NAME.format(threshold=skip_threshold)
    elif skip_threshold == 9:
        name = ASYMMETRIC_NAME.format(replace_threshold=replace_threshold)
    else:
        raise ValueError(
            f"No canonical Hybrid JSON for ({skip_threshold}, {replace_threshold})"
        )
    return result_dir / name


def load_design_point(
    path: Path,
    skip_threshold: int,
    replace_threshold: int,
    top2_ref: float | None,
) -> tuple[DesignPoint, float]:
    payload = load_json(path)
    hybrid_config = payload.get("hybrid_config")
    dewa_config = payload.get("dewa_config")
    if not isinstance(hybrid_config, dict) or not isinstance(dewa_config, dict):
        raise ValueError(f"Missing Hybrid/DEWA configuration in {path}")

    validate_protocol(payload, baseline, path)
    fp16_reference = float(payload.get("fp16_baseline_perplexity", np.nan))
    if not np.isclose(fp16_reference, baseline_ppl, rtol=0.0, atol=1e-12):
        raise ValueError(f"FP16 baseline mismatch in {path}")

    point_top2_baseline = float(payload.get("top2_baseline_perplexity", np.nan))
    if not np.isfinite(point_top2_baseline):
        raise ValueError(f"Missing Top-2 baseline in {path}")
    if top2_ref is None:
        top2_ref = point_top2_baseline
    elif not np.isclose(point_top2_baseline, top2_ref, rtol=0.0, atol=1e-12):
        raise ValueError(f"Top-2 baseline mismatch in {path}")

    perplexity = float(payload["perplexity"])
    return (
        DesignPoint(skip_threshold, replace_threshold, perplexity, path),
        top2_ref,
    )


design_points: list[DesignPoint] = []
gallery_top2: float | None = None
for skip_threshold, replace_threshold in DESIGN_POINTS:
    point, gallery_top2 = load_design_point(
        hybrid_result_path(skip_threshold, replace_threshold),
        skip_threshold,
        replace_threshold,
        gallery_top2,
    )
    design_points.append(point)
assert gallery_top2 is not None

x = np.arange(len(design_points), dtype=np.float64)
delta_ppl = np.asarray(
    [point.perplexity - gallery_top2 for point in design_points],
    dtype=np.float64,
)
tick_labels = [
    f"$({point.skip_threshold}, {point.replace_threshold})$"
    for point in design_points
]

fig, ax = plt.subplots(figsize=(3.85, 2.32))
ax.plot(x, delta_ppl, color="#E15759", linewidth=1.25, linestyle="-", zorder=3)
ax.scatter(
    x,
    delta_ppl,
    color="#E15759",
    marker="s",
    s=24,
    edgecolor="0.20",
    linewidth=0.55,
    zorder=4,
)
ax.axhline(0.0, color="0.38", linewidth=0.8, linestyle=(0, (4, 2)), zorder=1)
ax.set_xlabel(r"$(T_{\mathrm{skip}}, T_{\mathrm{replace}})$")
ax.set_ylabel(r"$\Delta$PPL from Top-2 A-BiE4")
ax.set_xticks(x)
ax.set_xticklabels(tick_labels, rotation=40, ha="right")
ax.set_xlim(float(x.min()) - 0.25, float(x.max()) + 0.25)
ax.set_ylim(-0.003, 0.060)
ax.set_yticks(np.arange(0.0, 0.0601, 0.010))
ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
style_axes(ax, grid_style="-")
ax.tick_params(axis="x", direction="out", pad=1.5)
ax.text(
    0.98,
    0.95,
    "Top-2 A-BiE4 baseline",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=7,
    color="0.30",
)
fig.tight_layout(pad=0.35)
save_figure(fig, FIGURES_G16 / "hybrid_tskip_treplace_delta_ppl")

Saved: C:\Users\user\Desktop\research\research1\plot\03_ppl-comparison\figures\g16\hybrid_tskip_treplace_delta_ppl.png


WindowsPath('C:/Users/user/Desktop/research/research1/plot/03_ppl-comparison/figures/g16/hybrid_tskip_treplace_delta_ppl.png')

## 4. Four-model Vanilla-BFP sweep (`vanilla_bfp_delta_ppl_g16/g32`)

Cross-model delta PPL for Vanilla BFP at Group sizes 16 and 32.

In [5]:
BITS = tuple(range(4, 9))
GROUP_SIZES = (16, 32)


@dataclass(frozen=True)
class ModelSpec:
    directory: str
    label: str
    color: str
    marker: str


@dataclass(frozen=True)
class VanillaResultPoint:
    bits: int
    perplexity: float
    fp16_perplexity: float
    path: Path


MODELS = (
    ModelSpec("llama2-7b", "LLaMA-2-7B", "#4C78A8", "o"),
    ModelSpec("llama2-13b", "LLaMA-2-13B", "#F28E2B", "s"),
    ModelSpec("llama3.1-8b", "LLaMA-3.1-8B", "#59A14F", "^"),
    ModelSpec("opt-6.7b", "OPT-6.7B", "#B279A2", "P"),
)


def find_single_result(directory: Path, bits: int, group_size: int) -> Path:
    matches = sorted(directory.glob(f"bfp{bits}-g{group_size}*.json"))
    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one BFP{bits}/G{group_size} result in {directory}; "
            f"found {len(matches)}"
        )
    return matches[0]


def discover_vanilla_results(
    spec: ModelSpec, group_size: int
) -> dict[int, VanillaResultPoint]:
    model_baseline = load_json(
        EXPERIMENTS_ROOT / "01_Baseline" / spec.directory / "baseline.json"
    )
    fp16_perplexity = float(model_baseline["perplexity"])
    result_dir = (
        EXPERIMENTS_ROOT / "02_Vanilla_BFP" / spec.directory / f"g{group_size}"
    )
    results: dict[int, VanillaResultPoint] = {}

    for bits in BITS:
        path = find_single_result(result_dir, bits, group_size)
        payload = load_json(path)
        validate_protocol(payload, model_baseline, path)
        config = payload.get("bfp_config")
        if not isinstance(config, dict):
            raise ValueError(f"Missing BFP configuration in {path}")
        if int(config.get("block_size", -1)) != group_size:
            raise ValueError(f"Group-size mismatch in {path}")
        if 1 + int(config.get("mantissa_bits", -1)) != bits:
            raise ValueError(f"BFP format mismatch in {path}")
        if not bool(config.get("quantize_lm_head", False)):
            raise ValueError(f"Vanilla BFP must quantize the LM head in {path}")
        perplexity = float(payload["perplexity"])
        results[bits] = VanillaResultPoint(bits, perplexity, fp16_perplexity, path)
    return results


vanilla_results = {
    group_size: {
        spec.directory: discover_vanilla_results(spec, group_size) for spec in MODELS
    }
    for group_size in GROUP_SIZES
}
x = np.asarray(BITS, dtype=np.float64)
deltas = [
    point.perplexity - point.fp16_perplexity
    for group_results in vanilla_results.values()
    for model_results in group_results.values()
    for point in model_results.values()
]
lower = min(-0.10, min(deltas) - 0.06)
upper = np.ceil((max(deltas) + 0.10) * 10.0) / 10.0

for group_size in GROUP_SIZES:
    fig, axis = plt.subplots(figsize=(3.5, 2.40))
    for spec in MODELS:
        model_results = vanilla_results[group_size][spec.directory]
        y = np.asarray(
            [
                model_results[bits].perplexity - model_results[bits].fp16_perplexity
                for bits in BITS
            ],
            dtype=np.float64,
        )
        axis.plot(
            x,
            y,
            color=spec.color,
            linewidth=1.25,
            marker=spec.marker,
            markersize=4.1,
            markerfacecolor=spec.color,
            markeredgecolor="0.20",
            markeredgewidth=0.55,
            label=spec.label,
            zorder=3,
        )
    axis.axhline(0.0, color="0.40", linewidth=0.85, linestyle="--", zorder=1)
    axis.set_xticks(x, [f"BFP{bits}" for bits in BITS])
    axis.set_xlim(BITS[0] - 0.15, BITS[-1] + 0.15)
    axis.set_ylim(lower, upper)
    axis.yaxis.set_major_locator(MultipleLocator(0.5))
    style_axes(axis)
    axis.set_xlabel("BFP format")
    axis.set_ylabel(r"$\Delta$PPL (vs. FP16)")
    axis.text(
        0.02,
        0.96,
        f"Group size = {group_size}",
        transform=axis.transAxes,
        ha="left",
        va="top",
    )
    axis.legend(
        loc="upper right",
        ncol=2,
        frameon=True,
        fancybox=True,
        framealpha=0.92,
        facecolor="white",
        edgecolor="0.75",
        handlelength=1.65,
        columnspacing=0.75,
        labelspacing=0.26,
        borderpad=0.45,
        borderaxespad=0.35,
    )
    fig.tight_layout(pad=0.35)
    save_figure(fig, FIGURES_G16_G32 / f"vanilla_bfp_delta_ppl_g{group_size}")

Saved: C:\Users\user\Desktop\research\research1\plot\03_ppl-comparison\figures\g16_g32\vanilla_bfp_delta_ppl_g16.png
Saved: C:\Users\user\Desktop\research\research1\plot\03_ppl-comparison\figures\g16_g32\vanilla_bfp_delta_ppl_g32.png
